# 2. Data Cleaning and Preparation

This notebook prepares the UCI Online Retail dataset for business analysis.

The workflow includes:
* Removing exact duplicate rows.
* Standardizing column names and data types.
* Preserving returns, cancellations, and unusual transactions for investigation.
* Creating revenue and date-related features.
* Saving the processed dataset separately from the original raw data.

The cleaning process is designed to be reproducible, with data-quality checks before and after cleaning.

In [1]:
# ==============================================================================
# LOAD THE ORIGINAL RETAIL DATASET FOR CLEANING
# ==============================================================================
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent

raw_file = project_root / "data" / "raw" / "Online Retail.xlsx"
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(raw_file)

print("Raw dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Raw dataset loaded successfully.
Rows: 541,909
Columns: 8


In [2]:
# ==============================================================================
# CREATE A WORKING COPY AND REMOVE EXACT DUPLICATE ROWS
# ==============================================================================
retail = df.copy()

rows_before = len(retail)

retail = retail.drop_duplicates().copy()

rows_after = len(retail)
duplicates_removed = rows_before - rows_after

print(f"Rows before cleaning: {rows_before:,}")
print(f"Duplicate rows removed: {duplicates_removed:,}")
print(f"Rows after deduplication: {rows_after:,}")

Rows before cleaning: 541,909
Duplicate rows removed: 5,268
Rows after deduplication: 536,641


In [3]:
# ==============================================================================
# STANDARDIZE COLUMN NAMES AND CONVERT FIELDS TO APPROPRIATE DATA TYPES
# ==============================================================================
retail.columns = retail.columns.str.strip()

retail["InvoiceNo"] = retail["InvoiceNo"].astype("string").str.strip()
retail["StockCode"] = retail["StockCode"].astype("string").str.strip()
retail["Description"] = retail["Description"].astype("string").str.strip()
retail["Country"] = retail["Country"].astype("string").str.strip()

retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], errors="coerce")
retail["Quantity"] = pd.to_numeric(retail["Quantity"], errors="coerce")
retail["UnitPrice"] = pd.to_numeric(retail["UnitPrice"], errors="coerce")
retail["CustomerID"] = pd.to_numeric(retail["CustomerID"], errors="coerce")

print(retail.dtypes)

InvoiceNo              string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                string
dtype: object


In [4]:
# ==============================================================================
# CREATE REVENUE, TRANSACTION FLAGS, AND DATE-RELATED FEATURES
# ==============================================================================
retail["Revenue"] = retail["Quantity"] * retail["UnitPrice"]

retail["IsCancellationInvoice"] = (
    retail["InvoiceNo"].str.startswith("C", na=False)
)

retail["IsNegativeQuantity"] = retail["Quantity"] < 0

retail["IsPositiveSale"] = (
    (retail["Quantity"] > 0) & (retail["UnitPrice"] > 0)
)

retail["Year"] = retail["InvoiceDate"].dt.year
retail["Month"] = retail["InvoiceDate"].dt.month
retail["YearMonth"] = retail["InvoiceDate"].dt.to_period("M").astype("string")
retail["Day"] = retail["InvoiceDate"].dt.day
retail["Hour"] = retail["InvoiceDate"].dt.hour

retail[[
    "InvoiceNo", "StockCode",
    "Quantity", "UnitPrice",
    "Revenue",
    "IsCancellationInvoice",
    "IsNegativeQuantity",
    "IsPositiveSale", "YearMonth",
    "Hour"
]].head()

,InvoiceNo,StockCode,Quantity,UnitPrice,Revenue,IsCancellationInvoice,IsNegativeQuantity,IsPositiveSale,YearMonth,Hour
0,536365,85123A,6,2.55,15.30,False,False,True,2010-12,8
1,536365,71053,6,3.39,20.34,False,False,True,2010-12,8
2,536365,84406B,8,2.75,22.00,False,False,True,2010-12,8
3,536365,84029G,6,3.39,20.34,False,False,True,2010-12,8
4,536365,84029E,6,3.39,20.34,False,False,True,2010-12,8


In [5]:
# ==============================================================================
# CHECK PROCESSED DATASET FOR REMAINING DUPLICATES AND INVALID ESSENTIAL FIELDS
# ==============================================================================
print("Remaining exact duplicates:", retail.duplicated().sum())
print("Missing InvoiceDate:", retail["InvoiceDate"].isna().sum())
print("Missing Quantity:", retail["Quantity"].isna().sum())
print("Missing UnitPrice:", retail["UnitPrice"].isna().sum())
print("Missing CustomerID:", retail["CustomerID"].isna().sum())
print("Rows with non-positive prices:", retail["UnitPrice"].le(0).sum())
print("Rows with negative quantities:", retail["Quantity"].lt(0).sum())

Remaining exact duplicates: 0
Missing InvoiceDate: 0
Missing Quantity: 0
Missing UnitPrice: 0
Missing CustomerID: 135037
Rows with non-positive prices: 2512
Rows with negative quantities: 10587


In [6]:
# ==============================================================================
# SAVE THE CLEANED TRANSACTION DATASET WITHOUT MODIFYING RAW EXCEL FILE
# ==============================================================================
output_file = processed_dir / "online_retail_clean.csv"

retail.to_csv(output_file, index=False)

print(f"Cleaned dataset saved to: {output_file}")
print(f"Final dataset shape: {retail.shape}")

Cleaned dataset saved to: c:\Users\saura\OneDrive\Documents\Data Science Projects\RetailPulse\data\processed\online_retail_clean.csv
Final dataset shape: (536641, 17)
